# LiTFiC — Carve a small BOBSL subset on Colab

Run this in **Colab with your Google Drive mounted** (the full BOBSL LMDBs live on your 5 TB Drive). It copies only a few whole episodes out of the 262 GB feature / pseudo-label LMDBs into a small folder you then upload to Kaggle.

**Runtime:** CPU is fine (no GPU needed). The notebook **auto-discovers** the data paths and **auto-picks** episodes — usually just **Run all**. You only touch a cell if auto-discovery guesses wrong (it prints `<-- FIX MANUALLY`).

If anything errors, paste the output back to Claude and it will fix the cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install lmdb

## CONFIG 1 — auto-discover the data on your Drive
This scans your Drive and fills the paths itself. The **only** thing you may adjust is `SEARCH_ROOT` (narrow it to the folder holding BOBSL so the scan is fast). If auto-discovery picks the wrong/multiple folders, override the marked lines manually.

In [ ]:
import os, glob

SEARCH_ROOT = '/content/drive/MyDrive'   # narrow to e.g. '/content/drive/MyDrive/bobsl' to speed up
OUT_DIR     = '/content/drive/MyDrive/bobsl_subset'

def _lmdb_dirs():
    return [os.path.dirname(m) for m in glob.glob(SEARCH_ROOT + '/**/data.mdb', recursive=True)]
def _pick(dirs, kws):
    return [d for d in dirs if any(k in os.path.basename(d).lower() for k in kws)]

dirs = _lmdb_dirs()
feats_hits = _pick(dirs, ['feat'])
pl_hits    = _pick(dirs, ['pl', 'pseudo', 'label'])
s2e_hits   = glob.glob(SEARCH_ROOT + '/**/subset2episode.json', recursive=True)

# auto-filled (first match) -- OVERRIDE any line if the guess is wrong:
FEATS_SRC = feats_hits[0] if feats_hits else 'SET_ME'
PL_SRC    = pl_hits[0]    if pl_hits    else 'SET_ME'
ORIG_SUBSET2EP = s2e_hits[0] if s2e_hits else 'SET_ME'
META_SRC  = os.path.dirname(ORIG_SUBSET2EP) if s2e_hits else 'SET_ME'

print('LMDB folders found:'); [print('  ', d) for d in dirs]
print('\nauto-selected:')
for name, val in [('FEATS_SRC', FEATS_SRC), ('PL_SRC', PL_SRC), ('ORIG_SUBSET2EP', ORIG_SUBSET2EP)]:
    ok = val != 'SET_ME' and os.path.exists(val)
    print(f'  {name:15}=', val, '' if ok else '  <-- FIX MANUALLY')
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# List available episodes per split so you can choose a few.
import json
with open(ORIG_SUBSET2EP) as f:
    orig = json.load(f)
for split, eps in orig.items():
    print(f"{split:12} {len(eps):5} episodes  e.g. {eps[:3]}")

## CONFIG 2 — episode selection (auto)
The next cell **auto-picks** 3 train + 1 eval episode from the real split file. Leave it as-is for the recommended subset, or edit the lists to choose specific episodes. **Whole episodes only** (previous-sentence context + per-episode DDP sharding need intact episodes).

In [ ]:
# --- EDIT THESE (use names exactly as printed) ---
TRAIN_EPS = orig['train'][:3]     # or list them explicitly: ['<ep1>', '<ep2>', '<ep3>']
VAL_EPS   = orig.get('val', orig['train'])[:1]
TEST_EPS  = VAL_EPS               # reuse val episode for a quick eval, or pick from a test split
# -------------------------------------------------
print('train:', TRAIN_EPS)
print('val  :', VAL_EPS)
print('test :', TEST_EPS)

In [ ]:
# Subset logic (same as scripts/subset_bobsl_lmdb.py, range-scan for speed on 262GB).
import lmdb

def _prefixes(episodes, kind):
    out = set()
    for ep in episodes:
        stem = os.path.splitext(ep)[0]
        if kind == 'feats':
            out.add(stem.encode('ascii'))
        else:
            out.add(f'{stem}_label'.encode('ascii'))
            out.add(f'{stem}_prob'.encode('ascii'))
    return out

def copy_subset(src_path, dst_path, episodes, kind, map_size=8*1024**3):
    wanted = _prefixes(episodes, kind)
    os.makedirs(dst_path, exist_ok=True)
    src = lmdb.open(src_path, readonly=True, lock=False, max_readers=512, subdir=os.path.isdir(src_path))
    dst = lmdb.open(dst_path, map_size=map_size, subdir=True)
    n = 0
    with src.begin() as rt, dst.begin(write=True) as wt:
        cur = rt.cursor()
        for prefix in sorted(wanted):
            seek = prefix + b'/'
            if not cur.set_range(seek):
                continue
            for key, val in cur:
                if not key.startswith(seek):
                    break
                wt.put(key, val); n += 1
    src.close(); dst.close()
    print(f'{kind}: copied {n} keys -> {dst_path}')
    return n

def write_subset2episode(out_dir, train, val, test):
    stems = lambda e: [os.path.splitext(x)[0] for x in e]
    m = {'train': stems(train), 'val': stems(val), 'public_test': stems(test), 'test': stems(test)}
    with open(os.path.join(out_dir, 'subset2episode.json'), 'w') as f:
        json.dump(m, f, indent=2)
    print('wrote subset2episode.json', {k: len(v) for k, v in m.items()})

In [ ]:
# Run the carve. All chosen episodes go into one LMDB pair; splits are defined by subset2episode.json.
all_eps = sorted(set(TRAIN_EPS) | set(VAL_EPS) | set(TEST_EPS))
copy_subset(FEATS_SRC, os.path.join(OUT_DIR, 'feats_lmdb'), all_eps, 'feats')
copy_subset(PL_SRC,    os.path.join(OUT_DIR, 'pl_lmdb'),    all_eps, 'pseudo-labels')
write_subset2episode(OUT_DIR, TRAIN_EPS, VAL_EPS, TEST_EPS)
!du -sh {OUT_DIR}/feats_lmdb {OUT_DIR}/pl_lmdb

In [ ]:
# Copy the small shared metadata (kept whole — it filters by episode at load time).
import shutil
META_FILES = [
    '8697_vocab.pkl',
    'info-src_videos_2236.pkl',
    'synonym_pickle_english_and_signdict_and_signbank.pkl',
    'best_delta_postpro_bobsl_best_traineval_0_pslab_ftune_.pkl',  # subtitles
    'manually-aligned.pkl',                                        # aligned subtitles (val/test)
    'blip2_captions_cleaned.pkl',
    'prev_gt_captions.json',
]
os.makedirs(os.path.join(OUT_DIR, 'meta'), exist_ok=True)
for fn in META_FILES:
    src = os.path.join(META_SRC, fn)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT_DIR, 'meta', fn)); print('copied', fn)
    else:
        print('SKIP (not found, adjust name):', fn)
!du -sh {OUT_DIR}

In [ ]:
# Zip the whole subset folder for a one-file upload to Kaggle (as a private Dataset).
!cd {OUT_DIR}/.. && zip -r -q bobsl_subset.zip bobsl_subset && du -sh bobsl_subset.zip
print('Download this zip from Drive, then create a private Kaggle Dataset from it.')

## Next steps
1. The subset (`feats_lmdb/`, `pl_lmdb/`, `subset2episode.json`, `meta/`) is on your Drive and zipped as `bobsl_subset.zip`.
2. Create a **private Kaggle Dataset** from the zip (Kaggle → Datasets → New → upload).
3. Continue with `docs/litfic-kaggle-runbook.md` (Stage 1 onward) to run the flow on 2×T4.

**Note:** `start_indices.json` for the subset is generated on Kaggle (Stage 3), after the paths point at this data.